<a href="https://colab.research.google.com/github/heerboi/AI-from-scratch/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Following Andrej's video: https://www.youtube.com/watch?v=kCc8FmEb1nY

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt

--2026-01-07 07:12:53--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M   150KB/s    in 6.8s    

2026-01-07 07:13:01 (159 KB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding="utf-8") as f:
    text = f.read()

In [3]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("Hii"))
print(decode(encode("Hii")))

[20, 47, 47]
Hii


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
split = int(0.9*len(data))
train_data = data[:split]
val_data = data[split:]
print(len(train_data))
print(len(val_data))

1003854
111540


In [8]:
#context length

block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(context, target)

tensor([18]) tensor(47)
tensor([18, 47]) tensor(56)
tensor([18, 47, 56]) tensor(57)
tensor([18, 47, 56, 57]) tensor(58)
tensor([18, 47, 56, 57, 58]) tensor(1)
tensor([18, 47, 56, 57, 58,  1]) tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) tensor(58)


In [10]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    x = x.to(device)
    y = y.to(device)
    return x, y

x, y = get_batch("train")
print(x)
print(y)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(num_embeddings = vocab_size, embedding_dim = vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B*T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # last time step for each batch and include all embeddings
            logits = logits[:, -1, :]

            probabilities = F.softmax(logits, dim=1)
            # (B, 1)
            next_idx = torch.multinomial(probabilities, num_samples=1)
            # (B, T+1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

m = BigramLanguageModel(vocab_size).to(device)
out, loss = m(x, y)
print(out.shape)
print(out)

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor([[ 1.6347, -0.0518,  0.4996,  ...,  0.2432,  1.1519,  0.9950],
        [ 0.3418, -0.9276,  1.2381,  ...,  1.5018, -0.5266,  0.2354],
        [ 0.1479, -0.4333,  0.5203,  ...,  0.3302,  1.5454,  1.3778],
        ...,
        [-0.5693, -0.0735,  0.7743,  ..., -0.0815, -1.1445, -0.0623],
        [ 0.4658, -0.2573, -1.0673,  ...,  1.2439,  1.3471,  1.6910],
        [-0.4553,  0.0139,  0.9309,  ...,  0.0290, -0.7568,  0.8701]],
       device='cuda:0', grad_fn=<ViewBackward0>)

yq$;tfBfROkNdcuwdZZTkOMl;,ertK
w:!PLCkMBbeA$3:XaSGJO-3p&M-c?KL3auhpFYVXJFhNNNuhq$OMxv.tbVFYdXlrFZaAe


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32

for steps in range(5000):
    xb,yb = get_batch('train')

    logits, loss = m(xb,yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.6424217224121094


In [15]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=1000)[0].tolist()))


Wawice my.

HDEdaromzy mu 
Yow&$LMuof isth ble mil;KI ll, ath iree sengmin lat HNGEdrovDEs, and Win nghire yWjus!
el lind me l.
lishe ce hiry ptug; aisspllw y.
Hllin's n Bfopetelives
MPOFGll, d mothakleo Windo whthCorib3MI'Tham dourive we hixend t so mower; te

ANk d nterurt f s ar igr Wam:

Enge maleronth,faf Pre?

WISo .
r f-NLLERar,

b&hak
Ardsal thes ghesthiuin cNI ayaraney Iry ts I&fr yES:
Myonge tonok,
I g.
AYor 'Wour me?
I
Tha anghy t-senomes twe meFlrdand s stz;

Whes th llety od,OThomuco ffvomy ssthecas l.
Tu Eias wethaleinju.
se eXJPeABene ovevLKimoCas!



Cos cok hedin tie s ind aus xVOFeRO, aLI:
Whit Clo gscPun?
WYUSis du he n,e, xme achZchitheakwhar
FRDurENINAs m s s withoumas Fond t sNTZlo INour id,ONF'sedInsurADYxI idurd pZ
&XnGinond Ca?
Fy
K:HBIUSHou tiund thornofen e sutan xaprythere whanothavitthers,lepEYBllellk t on s h O, t pr b.
Thwat d&Live Wout ir f; u;qyoeknen ouere 3fano iru fo.

FQS:WoQUEHRnk;G huchen tck is,, h pr t ftanofallon bay ho s, ag

AMe, meseveminds

In [16]:
eval_iters = 200
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out
estimate_loss()

{'train': tensor(2.5605), 'val': tensor(2.5711)}

## Mathematical trick in self-attention!

- have to average the logits in the time dim 0..t for logit t


In [17]:
B, T, C = 4, 8, 2
x = torch.randn(B,T,C)

In [18]:
div = torch.tril(torch.ones(T,T))
div /= div.sum(dim=1, keepdim=True)
xbow = div @ x

In [19]:
div

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [20]:
x[0], xbow[0]

(tensor([[ 1.4876,  0.7186],
         [-0.4844,  1.4378],
         [-0.8973,  0.2305],
         [ 0.1502,  0.3417],
         [ 0.2678,  1.4112],
         [ 1.7761,  0.7179],
         [ 1.0763, -0.8720],
         [ 1.0298,  0.3943]]),
 tensor([[1.4876, 0.7186],
         [0.5016, 1.0782],
         [0.0353, 0.7956],
         [0.0640, 0.6821],
         [0.1048, 0.8279],
         [0.3833, 0.8096],
         [0.4823, 0.5694],
         [0.5508, 0.5475]]))

### using softmax(infinity)

hint: e^-infinity = 0, and e^0 = 1

In [21]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=1)
xbow3 = wei @ x
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

## A bit about attention

- Attention is just a mechanism that adds a set of values with a set of weights. The approach above takes the weights to be equally distributed for the node itself and the nodes before, and zero for all nodes after.

- But, the current node might find more of what it needs from some nodes rather than others; it won't necessarily be equally distributed.

- Paper proposes an attention function where each node (token) at time T emits a query vector that contains the information that the current node is looking for, and a key vector that contains the information that the current node has within itself.

- This query vector and key vector get multiplied together to get the "affinities" between what the nodes are looking for and what the nodes have (T, T dimension, so each combination)

- Instead of taking the average of each node, we perform softmax on this new matrix. Now, instead of multiplying the "original" values $x$, we multiply it with the "value" matrix, which is different for each attention "head"

- As each head has a different purpose, it will have a different value to emit in each head, a different value that it posesses that makes more sense for that particular head!

In [22]:
head_size = 16
Q = nn.Linear(C, head_size, bias=False)
K = nn.Linear(C, head_size, bias=False)
V = nn.Linear(C, head_size, bias=False)

queries = Q(x)
keys = K(x)

print(queries.shape)
print(keys.shape)

torch.Size([4, 8, 16])
torch.Size([4, 8, 16])


In [23]:
T

8

In [24]:
tril = torch.tril(torch.ones(T, T))
wei = torch.einsum('btd,bad->bat', [queries, keys])
# print(wei)
# _wei = keys @ queries.transpose(-2, -1) # (4, 8, 8)
# wei = torch.zeros((T, T))
wei1 = wei.masked_fill(tril==0, float('-inf'))
wei1 = F.softmax(wei1, dim=1)
wei = F.softmax(wei,dim=1)

values = V(x)

xbow4 = wei @ values
xbow5 = wei1 @ values
print(wei.shape)
print(xbow4.shape)

torch.Size([4, 8, 8])
torch.Size([4, 8, 16])


In [25]:
wei[0], xbow4[0]

(tensor([[0.0172, 0.1552, 0.1926, 0.1170, 0.1229, 0.0095, 0.0143, 0.0344],
         [0.3273, 0.0124, 0.0175, 0.1224, 0.0629, 0.3627, 0.4419, 0.2907],
         [0.3863, 0.0227, 0.0205, 0.1383, 0.1070, 0.4186, 0.2890, 0.3132],
         [0.0952, 0.0575, 0.0577, 0.1296, 0.1197, 0.0756, 0.0707, 0.1154],
         [0.1156, 0.0265, 0.0387, 0.1179, 0.0718, 0.1011, 0.1470, 0.1378],
         [0.0116, 0.2059, 0.2599, 0.1152, 0.1287, 0.0058, 0.0094, 0.0259],
         [0.0179, 0.3897, 0.2728, 0.1364, 0.2528, 0.0091, 0.0067, 0.0333],
         [0.0290, 0.1300, 0.1402, 0.1231, 0.1341, 0.0176, 0.0209, 0.0493]],
        grad_fn=<SelectBackward0>),
 tensor([[ 2.1919e-01,  2.7197e-01,  2.8879e-01, -5.5813e-02,  2.2511e-02,
           1.8877e-01, -9.1920e-02,  1.5471e-01, -1.9074e-01, -3.3400e-01,
          -1.6355e-01,  2.9354e-01, -9.6372e-02, -1.4279e-01, -3.5971e-01,
          -2.1849e-01],
         [ 5.6546e-01,  9.2681e-01,  1.0634e+00,  7.0614e-02, -1.2326e-01,
           1.2807e+00, -2.8610e-01,  3.

In [26]:
wei1[0], xbow5[0]

(tensor([[0.0172, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3273, 0.0147, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3863, 0.0269, 0.0260, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0952, 0.0681, 0.0731, 0.2083, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1156, 0.0314, 0.0490, 0.1894, 0.1223, 0.0000, 0.0000, 0.0000],
         [0.0116, 0.2437, 0.3290, 0.1851, 0.2191, 0.1786, 0.0000, 0.0000],
         [0.0179, 0.4613, 0.3453, 0.2192, 0.4303, 0.2805, 0.2436, 0.0000],
         [0.0290, 0.1539, 0.1775, 0.1979, 0.2283, 0.5409, 0.7564, 1.0000]],
        grad_fn=<SelectBackward0>),
 tensor([[ 1.1065e-02,  1.6908e-02,  1.9073e-02,  2.1135e-04, -1.4229e-03,
           2.0732e-02, -5.3314e-03,  6.8205e-03,  1.2953e-02, -8.9847e-04,
          -1.6955e-02,  1.8341e-02,  5.0015e-03, -1.8611e-02, -1.2903e-03,
          -2.3593e-02],
         [ 2.1892e-01,  3.3170e-01,  3.7337e-01,  1.5023e-03, -2.5888e-02,
           4.0026e-01, -1.0487e-01,  1.

there's a little problem tho

In [27]:
query = torch.randn((4, 8, 16))
key = torch.randn((4, 8, 16))

print(query.var())
print(key.var())

tensor(0.9757)
tensor(0.9685)


In [28]:
qk = key @ query.transpose(-2, -1)
print(qk.var())

tensor(13.8549)


HUGE difference in variance, and when variance is high, means the difference between the values is huge. Since we'll apply softmax on this, if the values are very imbalanced, there'll be a huge imbalance in the weight assigned to other nodes, esp when the network is still untrained.

The paper proposes dividing the multiplication by the square root of head size, let's try it.

In [29]:
qk = key @ query.transpose(-2, -1) * head_size**-0.5
print(qk.var())

tensor(0.8659)


looks good

In [30]:
# num of attn heads running in parallel
n_heads = 16
# embedding size
# all layer final outputs must match 256
n_embd = 512

# individual heads are concat at the end
head_size = n_embd // n_heads

# size of ffn hidden layer
hidden_size = 1024

# total number of stacked transformer blocks
n_blocks = 6

block_size = 256

In [31]:
num = torch.arange(0, n_embd, 2).float()
thetas = 1.0/10000**(num/n_embd)

m = torch.arange(0, 5)
freqs1 = torch.einsum('i,j->ij', [m, thetas])
freqs2 = torch.outer(m, thetas).float()
print(freqs1, freqs2)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.0000e+00, 9.6466e-01, 9.3057e-01,  ..., 1.1140e-04, 1.0746e-04,
         1.0366e-04],
        [2.0000e+00, 1.9293e+00, 1.8611e+00,  ..., 2.2279e-04, 2.1492e-04,
         2.0733e-04],
        [3.0000e+00, 2.8940e+00, 2.7917e+00,  ..., 3.3419e-04, 3.2238e-04,
         3.1099e-04],
        [4.0000e+00, 3.8586e+00, 3.7223e+00,  ..., 4.4559e-04, 4.2984e-04,
         4.1465e-04]]) tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.0000e+00, 9.6466e-01, 9.3057e-01,  ..., 1.1140e-04, 1.0746e-04,
         1.0366e-04],
        [2.0000e+00, 1.9293e+00, 1.8611e+00,  ..., 2.2279e-04, 2.1492e-04,
         2.0733e-04],
        [3.0000e+00, 2.8940e+00, 2.7917e+00,  ..., 3.3419e-04, 3.2238e-04,
         3.1099e-04],
        [4.0000e+00, 3.8586e+00, 3.7223e+00,  ..., 4.4559e-04, 4.2984e-04,
         4.1465e-04]])


In [32]:
x = torch.randn(4, 8, 4)
print(x)
x[...,0]


tensor([[[ 2.3707e-01,  2.4512e-01, -1.8790e-01,  4.1169e-01],
         [ 1.1441e+00, -1.6672e-01,  5.1446e-01, -1.1556e+00],
         [ 2.7524e-02,  1.2105e+00, -1.5568e+00,  2.1462e+00],
         [-1.2993e+00, -4.9361e-01, -1.8776e-03,  5.4715e-02],
         [ 1.2378e+00,  4.3722e-01, -8.3715e-01,  8.3186e-01],
         [ 1.3422e+00,  1.2997e-02, -3.3048e-01,  2.9461e-01],
         [-5.9871e-01,  1.9335e+00, -7.5179e-02, -5.7792e-01],
         [-2.6852e+00,  3.6907e-01,  1.7484e+00, -1.2140e-01]],

        [[-2.9586e-01, -9.8793e-01, -1.0498e+00, -7.0688e-02],
         [-5.4958e-01,  9.7440e-01,  9.9558e-01, -3.0146e-01],
         [-1.8820e-01,  2.7321e-01, -1.5590e+00,  1.0395e+00],
         [-6.0283e-01,  1.2996e+00,  4.6471e-01,  9.7511e-01],
         [-5.1691e-04,  3.8398e-01,  1.7332e-01, -1.0522e+00],
         [-3.8466e-01,  9.2422e-01,  1.1591e+00, -8.7466e-01],
         [-1.4359e+00, -3.0357e+00, -8.3483e-02, -1.0368e+00],
         [ 9.6266e-01, -9.9735e-01,  2.8659e-02, -7.6

tensor([[ 2.3707e-01,  1.1441e+00,  2.7524e-02, -1.2993e+00,  1.2378e+00,
          1.3422e+00, -5.9871e-01, -2.6852e+00],
        [-2.9586e-01, -5.4958e-01, -1.8820e-01, -6.0283e-01, -5.1691e-04,
         -3.8466e-01, -1.4359e+00,  9.6266e-01],
        [ 9.8576e-01, -1.4931e+00,  2.9850e-01,  1.0317e+00,  1.9842e+00,
          1.2393e+00,  1.3809e-01, -1.2959e+00],
        [-1.3164e+00,  7.5862e-01,  8.3104e-01,  2.7391e-01,  1.3818e+00,
         -1.2075e+00, -1.1095e+00,  6.9579e-01]])

In [33]:
def create_thetas(seq_len, head_size, theta = 10000):

    num = torch.arange(0, head_size, 2).float()
    thetas = 1.0/theta**(num/head_size)

    m = torch.arange(0, seq_len)
    freqs = torch.einsum('i,j->ij', [m, thetas])
    # freqs = torch.outer(m, thetas).float()

    freqs_complex = torch.polar(torch.ones_like(freqs),freqs)

    return freqs_complex

def apply_rot_embd(x, freqs_complex):

    # all shapes except last; divide last shape into pairs of two
    # B,T,N,2
    x_mod = x.float().reshape(*x.shape[:-1], -1, 2)
    xr = x_mod[..., 0]
    xi = x_mod[..., 1]
    
    freqs_complex = freqs_complex.unsqueeze(0)
    cr = freqs_complex.real
    ci = freqs_complex.imag

    out_r = xr * cr - xi * ci
    out_i = xr * ci + xi * cr

    x_rot = torch.stack((out_r, out_i), dim=-1).reshape(*x.shape).to('cuda')
    return x_rot

In [34]:
class SingleAttentionHead(nn.Module):

    def __init__(self, rope_freqs, mask=False):
        super().__init__()
        self.Q = nn.Linear(n_embd, head_size, bias=False)
        self.K = nn.Linear(n_embd, head_size, bias=False)
        self.V = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.rope_freqs = rope_freqs
        # encoder attn module
        self.mask = mask

    def forward(self, x, encoder_embd=None):
        B, T, C = x.shape
        # (B, T, head_size)
        queries = apply_rot_embd(self.Q(x), self.rope_freqs)

        # in encoder arch, keys and values come from the encoder
        # this usually involves the ground truth
        if encoder_embd:
            keys = self.K(encoder_embd)
            values = self.V(encoder_embd)
        else:
            keys = self.K(x)
            values = self.V(x)


        keys = apply_rot_embd(keys, self.rope_freqs)

        wei = torch.einsum('btd, bad->bta', [queries, keys]) * head_size ** -0.5

        if self.mask:
            wei = wei.masked_fill(self.tril == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)

        x = wei @ values

        return x

class FFN(nn.Module):

    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.layers = nn.Sequential(
            # op: (B, T, hidden_size)
            nn.Linear(in_features, hidden_size, bias=bias),
            nn.GELU(),
            # op: (B, T, n_embd)
            nn.Linear(hidden_size, out_features, bias=bias),
        )
        self.layer_norm=nn.LayerNorm(out_features)

    def forward(self, x):
        out = x + self.layer_norm(self.layers(x))
        return out

In [35]:
class MultiAttentionBlock(nn.Module):

    def __init__(self, mask=False):
        super().__init__()


        rope_freqs = create_thetas(seq_len=block_size, head_size=head_size).to('cuda')

        self.heads = [SingleAttentionHead(mask=mask, rope_freqs=rope_freqs).to(device) for _ in range(n_heads)]

        self.linear = nn.Linear(n_embd, n_embd)

        self.layer_norm = nn.LayerNorm(n_embd)

    def forward(self, x, encoder_embd=None):
        # each op: (B, T, head_size)
        act = [head(x, encoder_embd) for head in self.heads]
        # op: (B, T, n_embd)
        out = x+self.layer_norm(self.linear(torch.concat(act, dim=-1)))

        return out

class DecoderBlock(nn.Module):
    def __init__(self, is_enc=False):
        super().__init__()

        self.multi_self_attention_block = MultiAttentionBlock(mask=True).to(device)
        if is_enc:
            self.cross_attn_block = MultiAttentionBlock(mask=False).to(device)
        self.ffn = FFN(n_embd, n_embd).to(device)

    def forward(self, x, encoder_embd=None):
        x = self.multi_self_attention_block(x)
        if encoder_embd:
            x = self.cross_attn_block(x, encoder_embd)
        x = self.ffn(x)

        return x

class EncoderBlock(nn.Module):
    def __init__(self):

        super().__init__()

        self.multi_self_attention_block = MultiAttentionBlock(mask=False).to(device)

        self.ffn = FFN(n_embd, n_embd).to(device)

    def forward(self, x):
        x = self.multi_self_attention_block(x)
        x = self.ffn(x)

        return x

In [36]:
def positional_embed(seq_len, n_embd):
    pe = torch.zeros(seq_len, n_embd, device=device)

    position = torch.arange(0, seq_len).unsqueeze(1).float()
    even = torch.arange(0,n_embd,2).float()

    pe[:, 0::2] = torch.sin(position / 10000**(2*even/n_embd))
    pe[:, 1::2] = torch.cos(position / 10000**((2*even+1)/n_embd))
    return pe

In [37]:
class Transformer(nn.Module):

    def __init__(self, encoder=False):
        super().__init__()
        self.encoder=encoder

        self.token_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)

        # positional embedding applied in the MultiAttentionBlock layer

        # self.rope_freqs = create_thetas(seq_len=block_size, n_embd=n_embd)
        # self.position_embedding_table = positional_embed(block_size, n_embd)
        # self.lm_head = SingleAttentionHead(head_size)
        # self.ffn = FFN(head_size, hidden_size)
        # self.attention_block = SingleAttentionBlock(head_size, hidden_size)

        # inp: (B, T, n_embd)
        # op:  (B, T, n_embd)
        # self.multi_head_attn = MultiAttentionBlock()

        # self.ffn = FFN(n_embd, n_embd)
        # pairs of multi head self attn blocks + ffn in sequence
        if encoder:
            self.encoder_block = nn.Sequential(*[EncoderBlock().to(device) for _ in range(n_blocks)])
        self.decoder_block = nn.Sequential(*[DecoderBlock(is_enc=encoder).to(device) for _ in range(n_blocks)])

        self.nn = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_embd = self.token_embedding_table(idx)
        # pos_embd = torch.nn.Dropout(0.1)(self.position_embedding_table)
        x = torch.nn.Dropout(0.1)(tok_embd)
        # x = self.lm_head(x)
        # x = self.ffn(x)
        # x = self.attention_block(x)
        # residual connections moved to their respective classes
        if self.encoder:
            x_enc = self.encoder_block(x)
            x = self.decoder_block(x, x_enc)
        else:
            x = self.decoder_block(x)

        logits = self.nn(x)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.view(B*T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets, label_smoothing=0.1)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # pick only last 8 tokens for next token
            idx_next = idx[:, -block_size:]
            logits, loss = self(idx_next)
            # last time step for each batch and include all embeddings
            logits = logits[:, -1, :]

            probabilities = F.softmax(logits, dim=1)
            # (B, 1)
            next_idx = torch.multinomial(probabilities, num_samples=1)
            # (B, T+1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

In [38]:
xb, yb = get_batch('train')

In [39]:
m = Transformer(encoder=False).to(device)
out, loss = m(xb, yb)
print(out.shape)
print(out)
print("Total parameters:")
print(sum([p.nelement() for p in m.parameters()]))

# print(decode(m.generate(torch.zeros((1,512), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))

torch.Size([8192, 65])
tensor([[-0.9373, -2.0909, -2.5574,  ..., -0.3803,  0.1362, -3.0016],
        [ 0.2328,  0.3781, -2.1764,  ...,  0.5654,  0.8075, -0.2809],
        [ 2.2782, -2.3604, -3.0196,  ...,  0.8101,  0.7979, -2.0831],
        ...,
        [ 0.1000, -1.4339, -2.4421,  ...,  1.6278, -0.0931, -1.2974],
        [-0.3300, -3.6348, -1.5458,  ..., -0.8667,  0.8447, -2.0318],
        [-1.1864, -2.9363, -2.3853,  ..., -0.0404, -0.8599, -3.2782]],
       device='cuda:0', grad_fn=<ViewBackward0>)
Total parameters:
7955521


In [40]:
optimizer = torch.optim.AdamW(m.parameters(), lr=5e-3)

In [41]:
batch_size = 64

for steps in range(10000):
    xb,yb = get_batch('train')

    logits, loss = m(xb,yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 100 == 0:
        print(f"Loss at {steps}: {loss.item()}")

Loss at 0: 6.603724479675293
Loss at 100: 2.86023211479187
Loss at 200: 2.827540874481201
Loss at 300: 2.821552276611328
Loss at 400: 2.825679063796997
Loss at 500: 2.8245151042938232
Loss at 600: 2.8145525455474854
Loss at 700: 2.7502880096435547
Loss at 800: 2.6558101177215576
Loss at 900: 2.6104347705841064
Loss at 1000: 2.469296932220459
Loss at 1100: 2.335988759994507
Loss at 1200: 2.2520103454589844
Loss at 1300: 2.164166212081909
Loss at 1400: 2.12758207321167
Loss at 1500: 2.0852463245391846
Loss at 1600: 2.0385336875915527
Loss at 1700: 2.0241434574127197
Loss at 1800: 1.9878573417663574
Loss at 1900: 1.9721980094909668
Loss at 2000: 1.9494242668151855
Loss at 2100: 1.9283809661865234
Loss at 2200: 1.9300901889801025
Loss at 2300: 1.91901695728302
Loss at 2400: 1.914670467376709
Loss at 2500: 1.884697675704956
Loss at 2600: 1.8676872253417969
Loss at 2700: 1.8732936382293701
Loss at 2800: 1.8761259317398071
Loss at 2900: 1.859635829925537
Loss at 3000: 1.8318564891815186
Loss 

KeyboardInterrupt: 

training a bit longer bec loss still decreasing

In [42]:
torch.save(m.state_dict(), "./model.pth")

In [ ]:
batch_size = 512

for steps in range(1500):
    xb,yb = get_batch('train')

    logits, loss = m(xb,yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps % 100 == 0:
        print(f"Loss at {steps}: {loss.item()}")

In [43]:
estimate_loss()

{'train': tensor(1.1530), 'val': tensor(2.7116)}

* transformer with single attention block, no layer norm, and no ffn: train 2.3442 val: 2.3719
* transformer with single attn, layer norm and ffn, no residual connection: train 2.2628 val 2.3023

* transformer with multi head attn + linear, layer norm, ffn and residual connection in after multihead attn: train 1.7584 val 1.9072

* transformer with multiple stacked attention-ffn blocks!: train 1.65 val 1.82

* using sinusoidal positional embedding converges much faster!! train 1.71 val 1.87

* GPU MAKES IT SM FASTERRRRRRR but model stops learnign because im doing layernorm after residual?? it works when i do residual after layernorm


new best: train 1.49 val 1.75

overfitting
new new best (07/01/26): train 1.15 val 2.711

In [44]:
print(decode(m.generate(torch.zeros((1,256), dtype=torch.long,device=device), max_new_tokens=1000)[0].tolist())[128:])

































































































































Wred kelrulatificipLeseme, not in to ze:
AReTuly a cXit, isZUBIs of at and while:Un forsworse posses,
cGor:
The Coriolve mock, what hath ourself t
SOMEAEtius boile's temere, wos it forVil:
And here :Yet of benef or onej, Me-rBILed hatms nourth,
Your hitMEREUTOest truchizS3 KONT.
Ok' it is
ZAPT. With other G--'twisdd! O, that fa'st jointced
As$oveHRKI heop3 KING OjuWARY:
sFouINket, unto thither, king;s, aif we Florence!
3 KITHAZNcIUS, God, rever him thY:
Why vorce a'ehclebve, like eanning towards grieve
For-own prUKmins Bufficia. Our mconHMoreLkest,
what th$ls with the great BaBuSjurupgh
As own yourAon-wzen: 's faunt o'llrWnxur's in?
Where were in himself he villain'd acWIARD:
Where is itwitted as the whitAHe, how itMu
H3 QetchForRQUEEN:
This iGORujumberle, that raVOLou:
Stream'st tht kinsmal ofher sawnBKath,
And vZvaluFI?wert thou soues of  bloo.
This iRpainMI

## positional encoding


In [ ]:
from math import sin, cos

In [ ]:
i = list(range(1,51))
pos = list(range(1,9))
embeddings = {n:[] for n in pos}
embeddings_no_div = {n:[] for n in pos}
for p in pos:
    for num in i:
        if num % 2 == 0:
            embed = sin(p)
            div_embed = sin(p/10000**(-2*num/512))
        else:
            embed = cos(p)
            div_embed = cos(p/10000**(-2*num/512))
        embeddings_no_div[p].append(embed)
        embeddings[p].append(div_embed)

In [ ]:
sin(1)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(20, 5)
ax[0].plot(embeddings[2])
ax[0].plot(embeddings_no_div[2], label = "Without division")
# ax[0].plot(embeddings[6])
ax[1].plot(embeddings[3])
ax[1].plot(embeddings_no_div[3], label = "without division")
# ax[1].plot(embeddings[5])
# plt.plot([e for e in embeddings.values()],label = [f"Pos{i}" for i in embeddings.keys()])
plt.legend()
plt.show()

In [ ]:
xb

In [ ]:
torch.arange(0, 5).unsqueeze(1).shape

In [ ]:
even = torch.arange(0,n_embd,2).float()
even+1

In [ ]:
def positional_embed(seq_len, n_embd):
    pe = torch.zeros(seq_len, n_embd)

    position = torch.arange(0, seq_len).unsqueeze(1).float()
    even = torch.arange(0,n_embd,2).float()

    pe[:, 0::2] = torch.sin(position / 10000**(2*even/n_embd))
    pe[:, 1::2] = torch.cos(position / 10000**((2*even+1)/n_embd))
    return pe

In [ ]:
positional_embed(8,8)